# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1.
2.
3.
4.
5.

In [5]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [6]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [7]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [8]:
# 1
months = np.array(["2026-09", "2026-10"])
sep = amount[clean & (month == months[0])].sum()
octo = amount[clean & (month == months[1])].sum()
change = (octo - sep) / sep * 100

print("строк в clean:", clean.sum())
print(f"выручка сентябрь: {sep:,.0f} ₸")
print(f"выручка октябрь:  {octo:,.0f} ₸")
print(f"изменение: {change:+.1f}%")

строк в clean: 4739
выручка сентябрь: 35,243,410 ₸
выручка октябрь:  34,419,520 ₸
изменение: -2.3%


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [9]:
# 2
cats = np.unique(cat)

rev = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months]
                for c in cats])
print(cats)
print(rev)

total = rev.sum(axis=1)
best = cats[total.argmax()]
print(f"\nбольше всего за два месяца: {best} ({total.max():,.0f} ₸)")

pct = (rev[:, 1] - rev[:, 0]) / rev[:, 0] * 100
for c, p in zip(cats, pct):
    print(f"{c}: {p:+.1f}%")
worst = cats[pct.argmin()]
print(f"\nсильнее всех упала: {worst} ({pct.min():+.1f}%)")

['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']
[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]

больше всего за два месяца: Электроника (34,370,430 ₸)
Дом: +2.6%
Книги: +7.1%
Красота: +2.6%
Одежда: -7.2%
Электроника: -2.5%

сильнее всех упала: Одежда (-7.2%)


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [10]:
# 3
cnt = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months]
                for c in cats])
print(cnt)

avg_check = rev / cnt
print()
for c, a in zip(cats, avg_check):
    print(f"{c}: сентябрь {a[0]:,.0f} ₸, октябрь {a[1]:,.0f} ₸")

top = cats[avg_check[:, 1].argmax()]
print(f"\nсамый высокий средний чек в октябре: {top} ({avg_check[:, 1].max():,.0f} ₸)")

[[476 476]
 [265 299]
 [422 435]
 [678 652]
 [531 505]]

Дом: сентябрь 10,499 ₸, октябрь 10,768 ₸
Книги: сентябрь 4,020 ₸, октябрь 3,816 ₸
Красота: сентябрь 6,187 ₸, октябрь 6,159 ₸
Одежда: сентябрь 13,521 ₸, октябрь 13,046 ₸
Электроника: сентябрь 32,773 ₸, октябрь 33,600 ₸

самый высокий средний чек в октябре: Электроника (33,600 ₸)


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [11]:
# 4

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [12]:
# 5

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [13]:
# 6

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [14]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [15]:
# 8